# 01 - Topology-3 inventory and regression test (workflow steps 1-2)

**Question:** is the validated hybrid CIGRE LV Topology-3 (Load Case 3, decentralized droop)
reproduced *exactly* in this project before anything is added to it?

Steps:
1. Inventory of every bus, line, transformer, VSC and load of the compiled PyDAE model, mapped
   against the network description of the paper.
2. Regression: re-run the original 32-h benchmark experiments (Case 1 / Case 3, with / without
   droop) with the unmodified model and compare with the CSV outputs of the original
   Topology-3 repository (see `external/README.md`).
3. Quasi-static check: the steady-state operating point used in this study (one `ini()` per
   15-min interval) equals the settled samples of the original time-domain run.
4. Base network operation (no BESS, no PV/wind) over the 24-h Case-3 day.

Outputs: `output/results/results_topology3_inventory.csv`, `output/results/results_regression.csv`,
`output/results/results_quasistatic_check.csv`, `output/results/results_base_network.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dfm_acdc.config import NetworkParams, RESULTS_DIR, FIG_DIR, TOPOLOGY3_DIR, RenewableParams, BESSParams
from dfm_acdc.network import Topology3, Injections
from dfm_acdc import plotstyle

plotstyle.apply_style()
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)
REF_DIR = os.path.join(TOPOLOGY3_DIR, "reference_outputs")   # regression outputs of the original repository
netp = NetworkParams()
net = Topology3(netp)

## 1. Inventory

In [ ]:
g = net.grid
rows = []
for b in g["buses"]:
    rows.append({"component": "bus", "name": b["name"], "detail": f"{b['U_kV']} kV"})
for t in g["transformers"]:
    rows.append({"component": "transformer", "name": f"{t['bus_j']}-{t['bus_k']}",
                 "detail": f"{t['S_n_kVA']} kVA, {t['U_j_kV']}/{t['U_k_kV']} kV, Rcc={t['R_cc_pu']}, Xcc={t['X_cc_pu']}, {t['connection']}"})
for l in g["lines"]:
    rows.append({"component": "DC line" if "dc" in l["code"] else "AC line", "name": f"{l['bus_j']}-{l['bus_k']}",
                 "detail": f"{l['code']}, {l['m']} m, Imax={g['line_codes'][l['code']]['I_max']} A"})
for v in g["vscs"]:
    ctrl = "Vdc-Q (DC-voltage forming)" if v["type"].endswith("vdc_q") else "P-Q with P-V droop"
    rows.append({"component": "VSC", "name": f"{v['bus_ac']}-{v['bus_dc']}",
                 "detail": f"{ctrl}, S_rated={netp.vsc_rating_kva[v['bus_ac']]} kVA"})
dfa, dfd = net.load.df_loads_ac, net.load.df_loads_dc
for bus, r in dfa.iterrows():
    ph = "3-ph" if bus in ("R01", "C01") else "1-ph (phase a)"
    rows.append({"component": "AC load (Case 3)", "name": bus, "detail": f"P={r['P (kW)']:.3f} kW, Q={r['Q (kvar)']:.3f} kvar, {ph}"})
for bus, r in dfd.iterrows():
    rows.append({"component": "DC load (Case 3)", "name": bus, "detail": f"P={r['Pdc (kW)']:.3f} kW"})
inv = pd.DataFrame(rows)
inv.to_csv(os.path.join(RESULTS_DIR, "results_topology3_inventory.csv"), index=False)
counts = inv.groupby("component").size()
print(counts.to_string())
print("\nAC buses:", len(net.ac_buses), " DC buses:", len(net.dc_buses))
print("VSCs:", [r["name"] + " : " + r["detail"] for r in rows if r["component"] == "VSC"])

Mapping against the paper: 40 AC buses (R01-R18, I01-I02, C01-C20), an 800-V meshed DC grid
of 26 buses (S-, H- and D-subgrids tied through S07-H02-D19), 3 transformers 20/0.4 kV
(500/150/300 kVA), 9 VSCs: **8 P-Q converters with identical P-V droop** at R01, R10, R14, I02,
C01, C09, C11, C16 (exactly the eight listed in Sec. 2.1) and **one Vdc-Q converter at I01-H01**
that forms the DC voltage (the "grid-forming interface"). DC lines: UG, 0.162 Ohm/km, 300 A.

## 2. Regression against the original outputs (bit-level)

In [ ]:
def v_lg(model, bus):   # original pydae.grid_tools 'V_an_m' == |V_phase_a| w.r.t. ground (node 0)
    return np.abs(model.get_values(f"V_{bus}_0_r") + 1j * model.get_values(f"V_{bus}_0_i"))

def currents_time_major(model):  # same nested order as the original export loop (t -> feeder -> phase)
    N = len(model.Time); out = []
    I = {(a, ph): np.hypot(model.get_values(f"i_t_MV0_{a}_2_{ph}_r"), model.get_values(f"i_t_MV0_{a}_2_{ph}_i"))
         for a in ("R01", "I01", "C01") for ph in range(3)}
    for t in range(N):
        for a in ("R01", "I01", "C01"):
            for ph in range(3):
                out.append(I[(a, ph)][t])
    return np.array(out)

reg = []
models = {}
for case, cn in (("Case 1", "case1"), ("Case 3", "case3")):
    for droop in (True, False):
        tag = "With_Droop" if droop else "Without_Droop"
        m = net.run_original_case(case, droop)
        models[(case, droop)] = m
        rv = pd.read_csv(os.path.join(REF_DIR, f"voltage_{tag}_T3_{cn}.csv"))
        dv = max(np.max(np.abs(v_lg(m, b) - rv[rv.Bus == b]["Voltage (V)"].values)) for b in rv.Bus.unique())
        ri = pd.read_csv(os.path.join(REF_DIR, f"current_{tag}_T3_{cn}.csv"))
        di = np.max(np.abs(currents_time_major(m) - ri["Current (A)"].values))
        reg.append({"case": case, "droop": droop, "n_samples": len(m.Time), "n_buses": rv.Bus.nunique(),
                    "max_abs_dV_V": dv, "max_abs_dI_A": di})
reg = pd.DataFrame(reg)
reg.to_csv(os.path.join(RESULTS_DIR, "results_regression.csv"), index=False)
print(reg.to_string())
assert (reg.max_abs_dV_V < 1e-9).all() and (reg.max_abs_dI_A < 1e-9).all(), "REGRESSION FAILED"
print("\nREGRESSION PASSED: original Topology-3 reproduced to machine precision.")

**Note on the voltage quantity.** The original notebooks' `gt.get_voltages(..., 'V_an_m')`
returns the phase-to-**ground** magnitude $|V_{a}|$ (node 0 w.r.t. the ground reference), not the
phase-to-neutral magnitude. Reproduction is exact only with that definition. In the present
study, AC voltage limits are evaluated on the physically relevant **phase-to-neutral** magnitude
$|V_{ph}-V_n|$ of all three phases (the stricter quantity); both are logged.

## 3. Quasi-static operating points vs. the original time-domain run

In [ ]:
m = models[("Case 3", True)]
T = m.Time.ravel()
rows = []
for h in range(0, 25):
    # In the original loop the large-step run reaches t + 2 s before the new loads are set, so the
    # loads of hour h (set at t = 3600 (h + 3) s) act during the following large-step interval and
    # the settled sample of hour h is the one at t = 3600 (h + 4) + 2 s.
    idx = int(np.argmin(np.abs(T - (3600 * (h + 4) + 2.0))))
    p = net.load_params(h)
    net.solve(p)
    me = net.measure()
    v_dyn = np.array([[abs(m.Y[idx, m.y_run_list.index(f"V_{b}_{ph}_r")] + 1j * m.Y[idx, m.y_run_list.index(f"V_{b}_{ph}_i")])
                       for ph in range(3)] for b in net.ac_buses])
    rows.append({"h": h, "t_sample_s": T[idx],
                 "max_abs_dV_lg_V": np.max(np.abs(v_dyn - me["V_lg"])),
                 "P_MV0_dyn_kW": m.Z[idx, m.outputs_list.index("p_MV0")] / 1e3, "P_MV0_qs_kW": me["P_MV0"]})
qs = pd.DataFrame(rows)
qs["dP_MV0_kW"] = qs.P_MV0_qs_kW - qs.P_MV0_dyn_kW
qs.to_csv(os.path.join(RESULTS_DIR, "results_quasistatic_check.csv"), index=False)
print(qs.round(6).to_string())
print("max |dV| = %.2e V, max |dP_MV0| = %.2e kW" % (qs.max_abs_dV_lg_V.max(), qs.dP_MV0_kW.abs().max()))

The one-state DAE (only the AGC integrator is dynamic) settles within each original hourly
interval, so the steady-state solution used here is the same operating point as the original
time-domain run; the remaining differences are solver-tolerance level. (Side finding: in the
original call sequence each hourly load set takes effect one large-step interval after it is
assigned, which is why the original plots apply a time shift; the operating points themselves
are unaffected.)

## 4. Base network operation (no BESS, no renewables), 15-min resolution

In [ ]:
rows = []
G = net.exceedance_groups()
for k in range(96):
    h = (k + 0.5) * 0.25
    p = net.load_params(h); net.solve(p); me = net.measure(); e = net.exceedance(me)
    r = {"t_h": k * 0.25, "P_MV0_kW": me["P_MV0"], "V_ln_min": me["V_ln"].min(), "V_ln_max": me["V_ln"].max(),
         "V_lg_min": me["V_lg"].min(), "V_lg_max": me["V_lg"].max(),
         "Vdc_min": me["V_dc"].min(), "Vdc_max": me["V_dc"].max(),
         "I_R01": me["I_feeder"][0].max(), "I_I01": me["I_feeder"][1].max(), "I_C01": me["I_feeder"][2].max(),
         "util_vsc_max": np.max(me["s_vsc"] / net._vsc_rating), "S_H01_kVA": me["s_vsc"][0], "e_max": e.max()}
    for gname, sl in G.items():
        r["e_" + gname] = e[sl].max()
    rows.append(r)
base = pd.DataFrame(rows)
base.to_csv(os.path.join(RESULTS_DIR, "results_base_network.csv"), index=False)
print(base.describe().T[["min", "max"]].round(3).to_string())
print("\nviolating intervals without BESS/renewables:", int((base.e_max > 1e-6).sum()), "of 96")

## Takeaway

The compiled Topology-3 model and the original load tool reproduce all four original 32-h
experiments to machine precision, and the quasi-static operating points used by this study
coincide with the settled samples of the original time-domain run. The network is therefore
used **unchanged**; the BESS, PV and wind of this study enter only as constant-power DC
injections superposed on the existing DC load inputs (exact, as the DC loads are constant-power).
Without storage and renewables the base case is feasible but tight: the R01 and C01 feeder heads
reach 97-99 % of their limits at the evening peak and the upper AC voltage is close to 1.05 pu.